# Stat Crew / Presto Football XML → CSV Converter

This notebook is a Python/Colab replacement for the old `GUIParseFootballXML` Java tool. It reads a Stat Crew / Presto Sports football play-by-play XML file (the same format the Java tool used) and produces the same 41-column CSV used to feed DV Sport.

**It is not just a straight port** — while rebuilding it, several real bugs in the original Java tool were found and fixed (by comparing its output against a manually-corrected boxscore). See the "Fixes vs. the original Java tool" section at the bottom for the full list.

**How to use this notebook:**
1. Run all the cells from top to bottom (Runtime → Run all).
2. When prompted, upload your `.xml` play-by-play file.
3. A CSV with the same name (and a summary of the game) will be generated and automatically downloaded.


## 1. Parser code
This cell defines the NFL-style team code lookup, the penalty-code translation table, the `Play`/`Player` data classes, and the main `parse()` function that walks the XML and builds one row per play.

In [ ]:
import xml.etree.ElementTree as ET
import sys

NFL_CODES = [
    ("alberta","CNAL"),("british","CNBC"),("calgary","CNCA"),("saskatchewan","CNSU"),
    ("manitoba","CNMT"),("regina","CNRG"),("montreal","CNMO"),("concordia","CNCO"),
    ("laval","CNLV"),("mcgill","CNMG"),("bishops","CNBI"),("sherbrooke","CNSH"),
    ("acadia","CNAC"),("francis","CNFX"),("allison","CNMA"),("saint","CNSM"),
    ("carleton","CNCR"),("mcmaster","CNMC"),("waterloo","CNWA"),("laurier","CNWI"),
    ("western","CNWO"),("queen","CNQN"),("ottawa","CNOT"),("toronto","CNTO"),
    ("windsor","CNWN"),("guelph","CNGU"),("york","CNYK"),("simon","CNSF"),
]

def nfl_code(name):
    n = name.lower()
    for key, code in NFL_CODES:
        if key in n:
            return code
    return ""

PENALTY_CODE_MAP = {
    "UNR": "UR",
}

def map_penalty_code(code):
    return PENALTY_CODE_MAP.get(code, code)


class Play:
    fields = ["gameKey","homeClubCode","awayClubCode","gameDate","quarter","possessionTeam",
        "typeOfPlay","series","seriesNum","gameClock","down","distance","runPass","fieldSide",
        "fieldPos","passResult","ballCarrier","gain","turnover","seriesBeg","seriesEnd",
        "specialTeamsPlayType","penaltyCode","penaltyYards","homeScore","awayScore","qbJerseyNum",
        "defender1","defender2","extraPoint","nullByPenalty","kickReturnYards","fumbleJerseyNum",
        "playSeq","homeScoreBefore","awayScoreBefore","kickLength","kickResult","kickerJerseyNum",
        "returnerJerseyNum","playDescription"]
    def __init__(self):
        for f in self.fields:
            setattr(self, f, "")
        self.playSeq = 0

    def row(self):
        return [self.gameKey,self.homeClubCode,self.awayClubCode,self.gameDate,self.possessionTeam,
        self.fieldSide,self.playSeq,self.quarter,self.typeOfPlay,self.series,self.seriesNum,
        self.gameClock,self.down,self.distance,self.runPass,self.fieldPos,self.passResult,
        self.ballCarrier,self.gain,self.turnover,self.seriesBeg,self.seriesEnd,
        self.specialTeamsPlayType,self.penaltyCode,self.penaltyYards,self.homeScore,self.awayScore,
        self.qbJerseyNum,self.defender1,self.defender2,self.extraPoint,self.nullByPenalty,
        self.kickReturnYards,self.fumbleJerseyNum,self.homeScoreBefore,self.awayScoreBefore,
        self.kickLength,self.kickResult,self.kickerJerseyNum,self.returnerJerseyNum,
        self.playDescription]

class Player:
    def __init__(self, vh, name, uni):
        self.vh = vh
        self.name = name
        self.uni = uni

def parse(xml_path):
    playList = []
    playerList = []
    currPlay = None
    isPlay = False

    quarterHolder = ""
    clockHolder = "15:00"
    homeScoreHolder = "0"
    awayScoreHolder = "0"
    homeScoreB4Holder = "0"
    awayScoreB4Holder = "0"
    playCounter = 0
    gameId = visId = homeId = visName = homeName = homeNFLCode = visNFLCode = gameDate = ""
    statCrewXMLVersion = "Unknown"
    rosterTeamId = ""
    homeSeries = visSeries = 0
    seriesTracker = ""
    homeSeriesNum = visSeriesNum = 0
    homeSeriesNumTracker = visSeriesNumTracker = ""
    specialTeamsTypeHolder = ""
    kickTeam = ""
    convertPending = False
    convertTeam = ""

    context = ET.iterparse(xml_path, events=("start","end"))

    for event, elem in context:
        tag = elem.tag
        if event == "start":
            if tag == "fbgame":
                statCrewXMLVersion = elem.get("version")
            if tag == "venue":
                gameId = elem.get("gameid")
                gameDate = elem.get("date")
                visId = elem.get("visid")
                homeId = elem.get("homeid")
                homeName = elem.get("homename")
                visName = elem.get("visname")
                homeNFLCode = nfl_code(homeName)
                visNFLCode = nfl_code(visName)
                playerList = []
            if tag == "play":
                clk = elem.get("clock")
                if clk is not None:
                    clockHolder = clk
                ptype = elem.get("type")
                text = elem.get("text")
                if ptype != "#" and ptype is not None:
                    isPlay = True
                    currPlay = Play()
                    kickTeam = ""
                    hasball = elem.get("hasball")
                    down = elem.get("down")
                    togo = elem.get("togo")
                    spot = elem.get("spot")
                    if hasball is not None and down is not None and togo is not None and spot is not None and text is not None:
                        currPlay.gameKey = gameId
                        currPlay.homeClubCode = homeNFLCode if homeNFLCode else homeId
                        currPlay.awayClubCode = visNFLCode if visNFLCode else visId
                        currPlay.gameDate = gameDate
                        currPlay.quarter = quarterHolder
                        if hasball == homeId:
                            currPlay.possessionTeam = homeNFLCode
                        else:
                            currPlay.possessionTeam = visNFLCode
                        if hasball == homeId:
                            currPlay.typeOfPlay = "OFF"
                            if ptype == "K":
                                currPlay.typeOfPlay = "KO"
                            elif ptype == "U":
                                currPlay.typeOfPlay = "P"
                            elif ptype in ("F", "X"):
                                # Default to FG; overridden to FGB below if the
                                # visiting team makes the kick (see p_fg/p_pat).
                                currPlay.typeOfPlay = "FG"
                        else:
                            currPlay.typeOfPlay = "DEF"
                            if ptype == "K":
                                currPlay.typeOfPlay = "KOR"
                            elif ptype == "U":
                                currPlay.typeOfPlay = "PR"
                            elif ptype in ("F", "X"):
                                currPlay.typeOfPlay = "FG"

                        isPenaltyOnly = text.startswith("PENALTY")

                        # A penalty flagged between a touchdown and its
                        # subsequent PAT/convert attempt is part of that kick
                        # sequence, not a normal scrimmage down: type it the
                        # same way a real convert attempt would be (FG/FGB),
                        # with no meaningful yards-to-go.
                        if isPenaltyOnly and convertPending and ptype == "E":
                            currPlay.typeOfPlay = "FG" if convertTeam == "OFF" else "FGB"

                        currPlay.series = ""
                        if currPlay.typeOfPlay in ("OFF","DEF") and not isPenaltyOnly:
                            if currPlay.typeOfPlay == "DEF" and currPlay.typeOfPlay != seriesTracker:
                                seriesTracker = currPlay.typeOfPlay
                                visSeries += 1
                            elif currPlay.typeOfPlay == "OFF" and currPlay.typeOfPlay != seriesTracker:
                                seriesTracker = currPlay.typeOfPlay
                                homeSeries += 1
                            if currPlay.typeOfPlay == "DEF":
                                currPlay.series = str(visSeries)
                            elif currPlay.typeOfPlay == "OFF":
                                currPlay.series = str(homeSeries)

                        currPlay.seriesNum = ""
                        if currPlay.typeOfPlay in ("OFF","DEF") and not isPenaltyOnly:
                            if currPlay.typeOfPlay == "DEF" and currPlay.series != visSeriesNumTracker:
                                visSeriesNumTracker = currPlay.series
                                visSeriesNum = 0
                            elif currPlay.typeOfPlay == "OFF" and currPlay.series != homeSeriesNumTracker:
                                homeSeriesNumTracker = currPlay.series
                                homeSeriesNum = 0
                            if currPlay.typeOfPlay == "DEF":
                                visSeriesNum += 1
                                homeSeriesNum = 0
                                currPlay.seriesNum = str(visSeriesNum)
                            elif currPlay.typeOfPlay == "OFF":
                                homeSeriesNum += 1
                                visSeriesNum = 0
                                currPlay.seriesNum = str(homeSeriesNum)

                        if clk is None:
                            currPlay.gameClock = clockHolder
                        else:
                            clockHolder = clk
                            currPlay.gameClock = clk
                        currPlay.down = down
                        currPlay.distance = togo

                        if spot.startswith(homeId):
                            currPlay.fieldSide = homeNFLCode
                            if currPlay.fieldSide == currPlay.possessionTeam:
                                currPlay.fieldPos = "-" + spot[len(homeId):]
                            else:
                                currPlay.fieldPos = spot[len(homeId):]
                        else:
                            currPlay.fieldSide = visNFLCode
                            if currPlay.fieldSide == currPlay.possessionTeam:
                                currPlay.fieldPos = "-" + spot[len(visId):]
                            else:
                                currPlay.fieldPos = spot[len(visId):]

                        # Normalize field position: drop leading zeros (e.g. "-01" -> "-1", "06" -> "6")
                        try:
                            currPlay.fieldPos = str(int(currPlay.fieldPos))
                        except ValueError:
                            pass

                        # "Goal to go" situations are recorded with togo="0" in the
                        # source data; the real yards-to-go is the distance from the
                        # line of scrimmage to the goal line, i.e. the field position.
                        # This doesn't apply to field goal/convert attempts (type F/X),
                        # where togo="0" is just a placeholder, not a real down-and-
                        # distance state.
                        if togo == "0" and ptype not in ("F", "X") and not (isPenaltyOnly and convertPending):
                            try:
                                currPlay.distance = str(abs(int(currPlay.fieldPos)))
                            except ValueError:
                                pass

                        if ptype == "F":
                            currPlay.specialTeamsPlayType = "FIELD GOAL"
                            specialTeamsTypeHolder = "FG"
                            if playList:
                                playList[-1].seriesEnd = specialTeamsTypeHolder.upper()
                        if ptype == "U":
                            currPlay.specialTeamsPlayType = "PUNT"
                            specialTeamsTypeHolder = "PUNT"
                            if playList:
                                playList[-1].seriesEnd = specialTeamsTypeHolder.upper()
                        if ptype == "K":
                            currPlay.specialTeamsPlayType = "KICK"
                            specialTeamsTypeHolder = "KICK"
                            if playCounter > 1 and playList:
                                playList[-1].seriesEnd = specialTeamsTypeHolder.upper()

                        if homeSeriesNum == 1 or visSeriesNum == 1:
                            if playList:
                                p = playList[-1]
                                if p.specialTeamsPlayType.upper() == "FIELD GOAL":
                                    currPlay.seriesBeg = "FG"
                                else:
                                    currPlay.seriesBeg = p.specialTeamsPlayType.upper()
                                if p.seriesEnd == "FUMBLE":
                                    currPlay.seriesBeg = p.seriesEnd

                        if "touchdown" in text.lower():
                            currPlay.seriesEnd = "TD"

                        currPlay.awayScoreBefore = awayScoreB4Holder
                        currPlay.homeScoreBefore = homeScoreB4Holder

                        # Touchdowns are worth 6 points and are scored
                        # immediately by the play itself (rather than waiting
                        # for a separate <score> tag, which may not appear
                        # until after a following PAT/penalty play).
                        if currPlay.seriesEnd == "TD":
                            if currPlay.typeOfPlay == "OFF":
                                homeScoreHolder = str(int(homeScoreHolder) + 6)
                            elif currPlay.typeOfPlay == "DEF":
                                awayScoreHolder = str(int(awayScoreHolder) + 6)
                            convertPending = True
                            convertTeam = currPlay.typeOfPlay

                        if ptype == "X":
                            # The actual PAT/convert attempt closes the window.
                            convertPending = False
                        elif not isPenaltyOnly and currPlay.seriesEnd != "TD":
                            # Any other real (non-penalty) play closes the window too.
                            convertPending = False

                        currPlay.awayScore = awayScoreHolder
                        currPlay.homeScore = homeScoreHolder

                        if "no play" in text.lower():
                            currPlay.nullByPenalty = "Y"
                            currPlay.runPass = "N"
                        else:
                            currPlay.nullByPenalty = "N"

                        playCounter += 1
                        currPlay.playSeq = playCounter

                        currPlay.playDescription = text
                    else:
                        currPlay.possessionTeam = "ERROR: Invalid/Out of Date File Format"
                else:
                    isPlay = False
                    # Certain administrative lines (e.g. timeouts) drop the
                    # "type" attribute entirely and are normally skipped. When
                    # they occur at a goal-to-go line (togo=="0"), the down/
                    # distance state is still meaningful, so a sparse row is
                    # still recorded (score, quarter, clock, down, distance,
                    # offense/defense, no play) even though there's no
                    # "official" play snapped.
                    hasball = elem.get("hasball")
                    down = elem.get("down")
                    togo = elem.get("togo")
                    spot = elem.get("spot")
                    if ptype is None and togo == "0" and hasball is not None and down is not None and spot is not None:
                        isPlay = True
                        currPlay = Play()
                        currPlay.quarter = quarterHolder
                        currPlay.typeOfPlay = "OFF" if hasball == homeId else "DEF"
                        if clk is None:
                            currPlay.gameClock = clockHolder
                        else:
                            clockHolder = clk
                            currPlay.gameClock = clk
                        currPlay.down = down
                        possessionTeam = homeNFLCode if hasball == homeId else visNFLCode
                        if spot.startswith(homeId):
                            fieldSide = homeNFLCode
                            fpos = spot[len(homeId):]
                        else:
                            fieldSide = visNFLCode
                            fpos = spot[len(visId):]
                        try:
                            currPlay.distance = str(abs(int(fpos)))
                        except ValueError:
                            currPlay.distance = ""
                        currPlay.runPass = "N"
                        currPlay.awayScore = awayScoreHolder
                        currPlay.awayScoreBefore = awayScoreB4Holder
                        currPlay.homeScore = homeScoreHolder
                        currPlay.homeScoreBefore = homeScoreB4Holder
                        playCounter += 1
                        currPlay.playSeq = playCounter

            if tag == "qtr":
                num = elem.get("number")
                if quarterHolder != num:
                    clockHolder = "15:00"
                quarterHolder = num

            # NOTE: the <score V=".." H=".."> tag that follows a scoring play
            # is intentionally NOT used to set scores here. It can appear one
            # or more plays after the play that actually scored (e.g. after a
            # PAT, or after an intervening penalty line), which would misattribute
            # the score change to the wrong row. Scores are instead applied
            # directly and immediately on the play that earns them (see the
            # touchdown handling above and the p_fg/p_pat handling below).

            if tag == "p_pa":
                result = elem.get("result")
                if result == "COMP":
                    currPlay.passResult = "C"
                elif result == "INT":
                    currPlay.passResult = "INT"
                    currPlay.turnover = "INT"
                    currPlay.seriesEnd = "INT"
                elif result in ("INC","DROP"):
                    currPlay.passResult = "I"
                elif result == "SACK":
                    currPlay.passResult = "S"
                else:
                    currPlay.passResult = result
                currPlay.runPass = "P"
                currPlay.qbJerseyNum = "ERR"
                qb = elem.get("qb")
                vh = elem.get("vh")
                for plyr in playerList:
                    if plyr.name.lower() == (qb or "").lower() and plyr.vh == vh:
                        currPlay.qbJerseyNum = plyr.uni
                        break
                if result in ("INC","SACK","INT","FUMB","DROP"):
                    currPlay.gain = "0"
                    currPlay.ballCarrier = ""
                else:
                    currPlay.gain = elem.get("gain")
                    # NOTE: rcv holds the receiver's jersey/uni number directly
                    # (Presto Sports schema), not a player name -- use it as-is.
                    currPlay.ballCarrier = elem.get("rcv")

            if tag == "p_ru":
                currPlay.gain = elem.get("gain")
                currPlay.ballCarrier = "ERR"
                currPlay.runPass = "R"
                name = elem.get("name")
                vh = elem.get("vh")
                for plyr in playerList:
                    if plyr.name.lower() == (name or "").lower() and plyr.vh == vh:
                        currPlay.ballCarrier = plyr.uni
                        break

            if tag == "p_pn":
                vh = elem.get("vh")
                code = map_penalty_code(elem.get("code"))
                if vh == "H":
                    currPlay.penaltyCode = currPlay.penaltyCode + homeNFLCode + " " + code + "; "
                else:
                    currPlay.penaltyCode = currPlay.penaltyCode + visNFLCode + " " + code + "; "
                yards = elem.get("yards")
                if yards is not None:
                    currPlay.penaltyYards = currPlay.penaltyYards + yards + "; "
                else:
                    currPlay.penaltyYards = ""
                if kickTeam != "" and yards is not None:
                    if kickTeam == vh:
                        currPlay.kickResult = str(int(currPlay.kickResult) - int(yards))

            if tag == "p_tk":
                assist = elem.get("assist")
                name = elem.get("name")
                vh = elem.get("vh")
                if assist == "Y":
                    if currPlay.defender1 == "":
                        currPlay.defender1 = "ERR"
                        for plyr in playerList:
                            if plyr.name.lower() == (name or "").lower() and plyr.vh == vh:
                                currPlay.defender1 = plyr.uni
                                break
                    else:
                        currPlay.defender2 = "ERR"
                        for plyr in playerList:
                            if plyr.name.lower() == (name or "").lower() and plyr.vh == vh:
                                currPlay.defender2 = plyr.uni
                                break
                else:
                    currPlay.defender1 = "ERR"
                    for plyr in playerList:
                        if plyr.name.lower() == (name or "").lower() and plyr.vh == vh:
                            currPlay.defender1 = plyr.uni
                            break

            if tag == "p_fumb":
                currPlay.fumbleJerseyNum = "ERR"
                frvh = elem.get("frvh")
                vh = elem.get("vh")
                if frvh != vh:
                    currPlay.turnover = "F"
                    currPlay.seriesEnd = "FUMBLE"
                name = elem.get("name")
                for plyr in playerList:
                    if plyr.name.lower() == (name or "").lower() and plyr.vh == vh:
                        currPlay.fumbleJerseyNum = plyr.uni
                        break

            if tag in ("p_ko","p_pu"):
                currPlay.kickLength = elem.get("gain")
                kickTeam = elem.get("vh")
                currPlay.kickResult = currPlay.kickLength
                currPlay.kickerJerseyNum = "ERR"
                name = elem.get("name")
                vh = elem.get("vh")
                for plyr in playerList:
                    if plyr.name.lower() == (name or "").lower() and plyr.vh == vh:
                        currPlay.kickerJerseyNum = plyr.uni
                        break

            if tag in ("p_kr","p_pr"):
                currPlay.kickReturnYards = elem.get("gain")
                vh = elem.get("vh")
                currPlay.kickResult = str(int(currPlay.kickLength) - int(currPlay.kickReturnYards))
                currPlay.returnerJerseyNum = "ERR"
                name = elem.get("name")
                for plyr in playerList:
                    if plyr.name.lower() == (name or "").lower() and plyr.vh == vh:
                        currPlay.returnerJerseyNum = plyr.uni
                        break

            if tag in ("p_fg", "p_pat"):
                # A successful field goal / convert kicked by the visiting
                # team is labeled FGB (vs. FG for the home team's kick).
                if elem.get("result") == "GOOD" and elem.get("vh") == "V":
                    currPlay.typeOfPlay = "FGB"

            if tag == "p_fg":
                # A made field goal is worth 3 points, applied immediately to
                # the kicking team's score on this same play (see note above
                # about not relying on the later <score> tag).
                if elem.get("result") == "GOOD":
                    if elem.get("vh") == "H":
                        homeScoreHolder = str(int(homeScoreHolder) + 3)
                    else:
                        awayScoreHolder = str(int(awayScoreHolder) + 3)
                    currPlay.homeScore = homeScoreHolder
                    currPlay.awayScore = awayScoreHolder
                elif currPlay.playDescription and "rouge" in currPlay.playDescription.lower():
                    # A missed field goal that goes into/through the end zone
                    # without being returned out earns the kicking team a
                    # single "rouge" point (USports/CIS rule).
                    if elem.get("vh") == "H":
                        homeScoreHolder = str(int(homeScoreHolder) + 1)
                    else:
                        awayScoreHolder = str(int(awayScoreHolder) + 1)
                    currPlay.homeScore = homeScoreHolder
                    currPlay.awayScore = awayScoreHolder

            if tag == "p_pat":
                result = elem.get("result")
                ptype2 = elem.get("type")
                if result == "GOOD":
                    if ptype2 == "K":
                        currPlay.extraPoint = "1"
                        currPlay.specialTeamsPlayType = "CONVERT(1)"
                        patPoints = 1
                    else:
                        currPlay.extraPoint = "2"
                        currPlay.specialTeamsPlayType = "CONVERT(2)"
                        patPoints = 2
                    # A made PAT/convert is applied immediately to the kicking
                    # team's score on this same play.
                    if elem.get("vh") == "H":
                        homeScoreHolder = str(int(homeScoreHolder) + patPoints)
                    else:
                        awayScoreHolder = str(int(awayScoreHolder) + patPoints)
                    currPlay.homeScore = homeScoreHolder
                    currPlay.awayScore = awayScoreHolder
                else:
                    currPlay.specialTeamsPlayType = "MISSED CONVERTM"

            if tag == "player":
                currPlayer = Player(rosterTeamId, elem.get("shortname"), elem.get("uni"))

            if tag == "plays":
                playList = []

            if tag == "team":
                rosterTeamId = elem.get("vh")

        elif event == "end":
            if tag == "play":
                if isPlay:
                    playList.append(currPlay)
                    # Sync the "before" score holders to the final score of
                    # this play, so the next play's *Before columns correctly
                    # reflect the score at the start of that next play.
                    homeScoreB4Holder = homeScoreHolder
                    awayScoreB4Holder = awayScoreHolder
            if tag == "player":
                playerList.append(currPlayer)
            elem.clear()

    return playList, statCrewXMLVersion



## 2. Upload your XML file and convert it

In [ ]:
from google.colab import files
import csv
import io
import os

uploaded = files.upload()

if not uploaded:
    raise SystemExit("No file uploaded.")

xml_filename = list(uploaded.keys())[0]
print(f"Parsing {xml_filename} ...")

plays, stat_crew_version = parse(xml_filename)
print(f"Found {len(plays)} plays. Stat Crew XML version: {stat_crew_version}")

base_name, _ = os.path.splitext(xml_filename)
csv_filename = base_name + ".csv"

HEADER = ["GAME KEY","HOME","VISIT","DATE","POSSESSION TEAM","FIELD SIDE","PLAY SEQ","QUARTER",
          "TYPE OF PLAY","SERIES","SERIES #","GAME CLOCK","DOWN","DISTANCE","RUN/PASS",
          "FIELD POSITION","PASS RESULT","BALL CARRIER","GAIN","TURNOVER","SERIES BEG","SERIES END",
          "SPECIAL TEAMS PLAY TYPE","PENALTY CODE","PENALTY YARDS","HOME CLUB SCORE","AWAY CLUB SCORE",
          "QB JERSEY #","DEFENDER 1","DEFENDER 2","EXTRA POINT","NULL BY PENALTY","KICK RETURN YARDS",
          "FUMBLE JERSEY #","HOME SCORE BEFORE","AWAY SCORE BEFORE","KICK LENGTH","KICK RESULT",
          "KICKER JERSEY #","RETURNER JERSEY #","PLAY DESCRIPTION"]

# newline="" + encoding="utf-8" avoids the mangled accented-character bug
# (e.g. "Piche" with an accent) that the original Java tool had.
with open(csv_filename, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["GUIParseFootballXML (Python/Colab port) Version: 2.0"])
    w.writerow(["Stat Crew XML Version: " + str(stat_crew_version)])
    w.writerow(HEADER)
    for p in plays:
        w.writerow(p.row())

print(f"Wrote {csv_filename}")
files.download(csv_filename)


## Fixes vs. the original Java tool

These were found by comparing the original tool's output against a manually-corrected boxscore, then tracing each discrepancy back to the XML:

1. **Ball carrier on completed passes** — the Java code looked up the receiver by matching the `rcv` attribute against player *names*, but `rcv` is actually already the receiver's jersey number in this schema. It now uses `rcv` directly.
2. **Penalty-only lines** — these used to be force-typed as `"PEN"` and excluded from `PLAY SEQ` numbering. They're now typed normally (`OFF`/`DEF`, same as any other play) and counted in `PLAY SEQ`, but still correctly excluded from series/series# numbering.
3. **FG vs. FGB** — not actually about offense/defense. A made field goal or PAT is `FGB` when the *visiting* team's kicker made it, and `FG` otherwise (home team's kick, or any non-good result for either team).
4. **Penalty code translation** — `UNR` (Unnecessary Roughness) in the XML is translated to the standard `UR` code, matching the rest of the penalty codes.
5. **Field position formatting** — leading zeros are stripped (`"06"` → `"6"`, `"-01"` → `"-1"`).
6. **"Goal to go" distance** — the source data uses `togo="0"` for goal-to-go downs; the real distance-to-go is the distance from the line of scrimmage to the goal line, not zero. (This does not apply to field goal/PAT attempts, where `togo="0"` is just a placeholder.)
7. **Timeouts at the goal line** — administrative "Timeout" lines have no `type` attribute and are normally skipped entirely. When one occurs at a goal-to-go line, it's still recorded as a sparse row (score/quarter/clock/down/distance only) since the down-and-distance state is meaningful.
8. **Scoring (the big one)** — the Java tool relied on a separate `<score V=".." H="..">` tag that can appear one or more plays *after* the play that actually scored (e.g. after the extra point, or after an intervening penalty), which misattributed the score change to the wrong row. Scores are now applied immediately on the scoring play itself: touchdown (+6), field goal (+3), PAT/convert (+1 or +2), and a missed field goal explicitly marked "rouge" in the text (+1, USports/CIS rule).
9. **Penalty during a PAT sequence** — a penalty flagged between a touchdown and its extra-point attempt is now typed the same way the extra point itself would be (`FG`/`FGB`) with distance `0`, instead of being treated as a normal down.
10. **Case-sensitive player name matching** — the roster lists the generic "team" player as `Team`, but play-by-play events sometimes reference it as `TEAM` (e.g. a kneel-down or an unassigned rush). Player-name lookups are now case-insensitive, so these resolve to `TM` instead of `ERR`.
11. **Encoding** — the CSV is now written as proper UTF-8, so accented names (e.g. "Piché") come out correctly instead of as mangled characters.

### Known limitation (not a parsing bug)
In the sample game used to validate this, two plays in the source XML were attributed to the wrong quarterback (a stat-crew data-entry error, not something detectable from the XML itself — `hasball`, `vh`, and `qb` are all internally consistent with the *wrong* player). Corrections like this can only be caught by a human reviewing the boxscore; they aren't something a generic parser can infer.
